In [1]:
from langgraph.graph import StateGraph, START, MessagesState
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages.utils import trim_messages,count_tokens_approximately

/home/amir/LangGraph/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
model = ChatGroq(model_name="llama-3.3-70b-versatile")

In [4]:
MAX_TOKENS = 150

In [5]:
def call_model(state: MessagesState):

    messages = trim_messages(
        state["messages"],
        strategy="last",                      
        token_counter=count_tokens_approximately,
        max_tokens=MAX_TOKENS
    )

    print('Current Token Count ->', count_tokens_approximately(messages=messages))

    for message in messages:
        print(message.content)

    response = model.invoke(messages)

    return {"messages": [response]}

In [6]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")

In [7]:
checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [8]:
config = {"configurable": {"thread_id": "1"}}

result = graph.invoke(
    {"messages": [{"role": "user", "content": "Hi, my name is Amir."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 9
Hi, my name is Amir.


"Hello Amir, it's nice to meet you. Is there something I can help you with or would you like to chat?"

In [9]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "I am learning ai/ml."}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 49
Hi, my name is Amir.
Hello Amir, it's nice to meet you. Is there something I can help you with or would you like to chat?
I am learning ai/ml.


"That's a fascinating field. AI and ML are constantly evolving and have numerous applications in various industries. What specific areas of AI/ML are you interested in or currently learning about? Are you working on any projects or using any particular tools and frameworks like TensorFlow, PyTorch, or scikit-learn?"

In [ ]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "Can you explain short term memory?"}]},
    config,
)

result["messages"][-1].content
# here the token exceed so we trim it

Current Token Count -> 146
Hi, my name is Amir.
Hello Amir, it's nice to meet you. Is there something I can help you with or would you like to chat?
I am learning ai/ml.
That's a fascinating field. AI and ML are constantly evolving and have numerous applications in various industries. What specific areas of AI/ML are you interested in or currently learning about? Are you working on any projects or using any particular tools and frameworks like TensorFlow, PyTorch, or scikit-learn?
Can you explain short term memory?


'Short-term memory (STM) refers to the ability to hold and manipulate information in your mind for a short period of time, typically ranging from a few seconds to a minute. It\'s a fundamental concept in cognitive psychology and is closely related to working memory.\n\nHere\'s how it works:\n\n1. **Information enters**: You perceive information through your senses, such as seeing, hearing, or reading.\n2. **Encoding**: The information is encoded into a format that your brain can understand, like a mental representation of a phone number or a word.\n3. **Holding**: The encoded information is held in your short-term memory for a brief period, usually around 15-30 seconds.\n4. **Manipulation**: You can manipulate the information in your short-term memory, such as repeating a phone number to yourself or doing mental math.\n5. **Decay or retrieval**: If you don\'t rehearse or repeat the information, it will decay and be forgotten. However, if you repeat it or connect it to something meaning

In [11]:
result = graph.invoke(
    {"messages": [{"role": "user", "content": "What is my name?"}]},
    config,
)

result["messages"][-1].content

Current Token Count -> 8
What is my name?


"I don't know your name. I'm a large language model, I don't have the ability to know your personal information or recall previous conversations. Each time you interact with me, it's a new conversation. If you'd like to share your name, I'd be happy to chat with you!"

In [16]:
for item in graph.get_state({"configurable": {"thread_id": "1"}}).values['messages']:
    print(item.content)
    print('-'*120)



Hi, my name is Amir.
------------------------------------------------------------------------------------------------------------------------
Hello Amir, it's nice to meet you. Is there something I can help you with or would you like to chat?
------------------------------------------------------------------------------------------------------------------------
I am learning ai/ml.
------------------------------------------------------------------------------------------------------------------------
That's a fascinating field. AI and ML are constantly evolving and have numerous applications in various industries. What specific areas of AI/ML are you interested in or currently learning about? Are you working on any projects or using any particular tools and frameworks like TensorFlow, PyTorch, or scikit-learn?
------------------------------------------------------------------------------------------------------------------------
Can you explain short term memory?
----------------------